# Explore Fraud and Delayed Labels

FraudTwin keeps the simulator's ground truth separate from what an operational fraud system can observe. This tutorial explores fraud scenarios, hard negatives, alerts, cases, and delayed labels.

## 1. Generate fraud-enabled data

This tutorial uses the fraud-enabled benchmark configuration. It generates fraud and legitimate lookalikes so the two populations can be compared.

In [1]:
# ruff: noqa
from pathlib import Path

import polars as pl

import fraudtwin

project_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "configs" / "minimal.yaml").is_file()
)
config_path = project_root / "configs" / "benchmarks" / "m12-difficulty-v1.yaml"
data = fraudtwin.generate(config_path)

print("Run:", data.run_id)

Run: RUN-642709d737867d7f


## 2. Compare fraud records

A fraud record describes simulator truth. Hard negatives are legitimate records deliberately generated to resemble fraud.

In [2]:
# ruff: noqa
fraud_records = pl.DataFrame(
    [
        {
            "Record type": record.record_type,
            "Fraud truth": record.fraud_truth,
            "Scenario": record.scenario_type,
            "Payment": record.payment_id,
            "Amount": record.amount,
        }
        for record in data.behavior.fraud_records
    ]
)

fraud_records.group_by(["Record type", "Fraud truth"]).len().rename({"len": "Records"})

Record type,Fraud truth,Records
str,bool,u32
"""HARD_NEGATIVE""",false,9
"""FRAUD""",true,25


## 3. Inspect what becomes observable

Alerts and cases are operational observations derived from fraud records. They are not the same as the simulator's original fraud truth.

In [3]:
# ruff: noqa
workflow_counts = pl.DataFrame(
    {
        "Artifact": ["Fraud records", "Alerts", "Cases", "Labels"],
        "Rows": [
            len(data.behavior.fraud_records),
            len(data.behavior.alerts),
            len(data.behavior.fraud_cases),
            len(data.behavior.fraud_labels),
        ],
    }
)

workflow_counts

Artifact,Rows
str,i64
"""Fraud records""",34
"""Alerts""",34
"""Cases""",34
"""Labels""",34


## 4. See label delay

A label becomes available after the fraud is observed and investigated. The delay is why future information must not be used when constructing historical ML data.

In [4]:
# ruff: noqa
labels = pl.DataFrame(
    [
        {
            "Payment": label.payment_id,
            "Label": label.label,
            "Fraud occurred": label.fraud_occurred_at,
            "Label available": label.label_available_at,
        }
        for label in data.behavior.fraud_labels[:5]
    ]
)

labels

Payment,Label,Fraud occurred,Label available
str,str,"datetime[μs, UTC]","datetime[μs, UTC]"
"""PAY-F01-000001-000001""","""FRAUD""",2026-01-01 00:00:00 UTC,2026-01-03 01:06:11 UTC
"""PAY-F01-000001-000002""","""FRAUD""",2026-01-01 00:00:02 UTC,2026-01-03 01:06:13 UTC
"""PAY-F01-000001-000003""","""FRAUD""",2026-01-01 00:00:03 UTC,2026-01-03 01:06:14 UTC
"""PAY-HN-F01-000001-000001""","""LEGITIMATE""",2026-01-01 00:00:00 UTC,2026-01-02 01:06:07 UTC
"""PAY-HN-F01-000004-000001""","""LEGITIMATE""",2026-01-01 00:00:00 UTC,2026-01-02 01:06:07 UTC


FraudTwin now shows the complete observation path: fraud truth is generated first, workflow artifacts appear later, and labels become available only after their configured delay.

## Record the generated shape and tutorial contract.


In [ ]:
# ruff: noqa
active = next(
    (globals().get(name) for name in ("data", "baseline") if globals().get(name) is not None), None
)
assert active is not None
summary = {
    "tutorial_id": 4,
    "payments": len(active.behavior.payments),
    "events": len(active.behavior.payment_events),
}
print(summary)
assert summary["payments"] >= 0

## Inspect stable payment identities.


In [ ]:
# ruff: noqa
ids = [item.payment_id for item in active.behavior.payments]
assert len(ids) == len(set(ids))
print({"unique_payment_ids": len(ids)})

## Compare event-time coverage.


In [ ]:
# ruff: noqa
times = [event.event_time for event in active.behavior.payment_events]
print({"first_event": min(times) if times else None, "last_event": max(times) if times else None})

## Record the generated shape and tutorial contract.


In [ ]:
# ruff: noqa
active = next(
    (globals().get(name) for name in ("data", "baseline") if globals().get(name) is not None), None
)
assert active is not None
summary = {
    "tutorial_id": 4,
    "payments": len(active.behavior.payments),
    "events": len(active.behavior.payment_events),
}
print(summary)
assert summary["payments"] >= 0

## Inspect stable payment identities.


In [ ]:
# ruff: noqa
ids = [item.payment_id for item in active.behavior.payments]
assert len(ids) == len(set(ids))
print({"unique_payment_ids": len(ids)})

## Compare event-time coverage.


In [ ]:
# ruff: noqa
times = [event.event_time for event in active.behavior.payment_events]
print({"first_event": min(times) if times else None, "last_event": max(times) if times else None})